# 06. Qdrant 기반 Naive RAG

## 학습 목표

- PDF 청크를 임베딩하여 Qdrant에 저장한다.
- 사용자 질문과 유사한 문단을 검색한다.
- 검색 문단을 LLM 답변 생성에 연결한다.

## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

> 이 노트북은 `src` 파일을 import하지 않고, 노트북 안의 코드만으로 실행되도록 구성한다.

In [1]:
from pathlib import Path
import re
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True, dotenv_path="../../.env")

PDF_PATH = Path("../data/공직자_민원응대_핵심_매뉴얼.pdf")

## 1. PDF 로딩 + 청크 분할 함수

In [2]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_pdf_pages(pdf_path: Path) -> list[Document]:
    reader = PdfReader(str(pdf_path))
    docs = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if text:
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": pdf_path.name,
                        "page": page_number
                    }
                )
            )

    return docs


def split_docs(docs: list[Document]) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=120,
        separators=["\n\n", "\n", "STEP", "‣", "•", ". ", " "]
    )
    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i

    return chunks

## 2. 문서 청크 생성

In [3]:
pages = load_pdf_pages(PDF_PATH)
chunks = split_docs(pages)

print("페이지 수:", len(pages))
print("청크 수:", len(chunks))

페이지 수: 20
청크 수: 43


## 3. Qdrant 벡터스토어 생성

실습 편의를 위해 Qdrant를 인메모리 모드로 사용한다.

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="civil_service_manual",
    force_recreate=True
)

print("Qdrant 저장 완료")

Qdrant 저장 완료


## 4. 유사 문서 검색

In [5]:
question = "민원인이 반복 전화를 하면 어떻게 해야 하나요?"

retrieved_docs = vectorstore.similarity_search(question, k=4)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- 검색 결과 {i} ---")
    print(doc.metadata)
    print(doc.page_content[:600])

--- 검색 결과 1 ---
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 7, 'chunk_id': 7, '_id': '47c8d44359634fd187c5331e87d59b07', '_collection_name': 'civil_service_manual'}
12 | 현장공무원을 위한 민원응대 핵심매뉴얼
03 전화민원
• 민원인에게 인사한 후, 소속 등을 밝힘
STEP
1
• 민원인의 문의사항을 정확하게 판단하여 상담 진행
STEP
2
• 민원상담이 끝나면 인사 후에 전화 종료
 ※ 전화로 접수·처리가 불가능한 민원을 제기한 경우에는 서면 등(인터넷·팩스 등 
 정보통신망 또는 우편)으로 제출하거나 방문하도록 유도
STEP
3
 민원인을 배려하는 표현
‣ 기다리세요.
‣ 모르겠는데요.
‣ 자리에 없습니다.
‣ 예? 뭐라고요?
‣ 아닙니다.
‣ 담당자가 아니라서 모르겠는데요.
‣ 할 수 없는데요.
‣ 잠시만 기다려 주시겠습니까?
‣ 확인 후 알려드려도 될까요?
‣ 지금 자리에 안 계십니다. 메모를 남겨드릴까요?
‣ 다시 한번 말씀해 주시겠습니까?
‣ 제가 확인한 결과 00가 아니라고 합니다. 
 양해해 주시면 감사하겠습니다.
‣ 담당자가 아니라서 제가 잘 알지 못하는 내용이니, 전화를 돌려드려도 
 되겠습니까? 담당자의 전화번호는 000-0000입니다.
‣ 여기서는 해결해 드릴 수 없는 민원입니다. ooo으로 가셔서 문의를 
 해보시기 바랍니다.
‣ 현재 규정으로는 할 수가 없습니다. (가능할 경우) 혹시 도울 수
--- 검색 결과 2 ---
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 12, 'chunk_id': 22, '_id': 'eda93517c21647e39983b4b9a927ff8f', '_collection_name': 'civil_service_manual'}
바랍니다.
[ 경고문구(예시) ]
정당한 사유없는 반복민원 ※ 민원처리법 제23조, 제34조4-2
‣ 기관별 민원조정위원회 심의

## 5. 검색 결과를 프롬프트에 넣기

In [6]:
def format_docs(docs: list[Document]) -> str:
    formatted = []

    for i, doc in enumerate(docs, start=1):
        formatted.append(
            f"[근거 {i}] page={doc.metadata.get('page')}, chunk={doc.metadata.get('chunk_id')}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted)


context = format_docs(retrieved_docs)
print(context[:1500])

[근거 1] page=7, chunk=7
12 | 현장공무원을 위한 민원응대 핵심매뉴얼
03 전화민원
• 민원인에게 인사한 후, 소속 등을 밝힘
STEP
1
• 민원인의 문의사항을 정확하게 판단하여 상담 진행
STEP
2
• 민원상담이 끝나면 인사 후에 전화 종료
 ※ 전화로 접수·처리가 불가능한 민원을 제기한 경우에는 서면 등(인터넷·팩스 등 
 정보통신망 또는 우편)으로 제출하거나 방문하도록 유도
STEP
3
 민원인을 배려하는 표현
‣ 기다리세요.
‣ 모르겠는데요.
‣ 자리에 없습니다.
‣ 예? 뭐라고요?
‣ 아닙니다.
‣ 담당자가 아니라서 모르겠는데요.
‣ 할 수 없는데요.
‣ 잠시만 기다려 주시겠습니까?
‣ 확인 후 알려드려도 될까요?
‣ 지금 자리에 안 계십니다. 메모를 남겨드릴까요?
‣ 다시 한번 말씀해 주시겠습니까?
‣ 제가 확인한 결과 00가 아니라고 합니다. 
 양해해 주시면 감사하겠습니다.
‣ 담당자가 아니라서 제가 잘 알지 못하는 내용이니, 전화를 돌려드려도 
 되겠습니까? 담당자의 전화번호는 000-0000입니다.
‣ 여기서는 해결해 드릴 수 없는 민원입니다. ooo으로 가셔서 문의를 
 해보시기 바랍니다.
‣ 현재 규정으로는 할 수가 없습니다. (가능할 경우) 혹시 도울 수 있는 
 다른 방법이 있는지 확인해 보겠습니다.
무심코 쓰는 표현 민원인을 배려하는 표현
1. 공통사항 15
2. 전화응대 15
3. 대면응대 18

[근거 2] page=12, chunk=22
바랍니다.
[ 경고문구(예시) ]
정당한 사유없는 반복민원 ※ 민원처리법 제23조, 제34조4-2
‣ 기관별 민원조정위원회 심의 
 종결 처리 후 다시 접수된 반복민원 중 처리주무부서장이 신중한 
 처리가 필요하다고 판단, 기관별 민원조정위원회 심의로 종결
 위원회 의견 등 결정근거 및 이의제기 절차 등 구체적 안내
 ※ 고충민원(권익위·지방옴부즈만), 거부처분 이의신청(행정심판·소송) 등 안내
‣ 지도·감독등 민원조정위원회 심의 
 심의결과 통보 후 다시 접수된 반복민원 

## 6. LLM 답변 생성

In [7]:
prompt = ChatPromptTemplate.from_messages([
    ("system", '''
당신은 공직자 민원응대 매뉴얼 기반 업무지원 AI입니다.
반드시 제공된 근거 문단 안의 내용만 사용해 답변하세요.
근거에 없는 내용은 추측하지 마세요.

답변 형식:
1. 핵심 대응
2. 단계별 조치
3. 사용할 수 있는 안내 표현
4. 근거
5. 주의사항
'''),
    ("human", '''
질문:
{question}

근거 문단:
{context}

위 근거를 바탕으로 답변하세요.
''')
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = prompt | llm | StrOutputParser()

answer = rag_chain.invoke({
    "question": question,
    "context": context
})

print(answer)

1. 핵심 대응  
반복 전화를 하는 민원인에게는 충분한 설명을 제공한 후, 다른 민원 처리를 위해 통화를 종료해야 합니다.

2. 단계별 조치  
STEP 1: 민원인에게 충분한 설명을 제공합니다.  
STEP 2: 반복 전화가 계속될 경우, "현재 처리 중에 있습니다." 또는 "정해진 규정과 절차에 따라 정상적으로 처리된 사항입니다."라고 안내합니다.  
STEP 3: 통화 지속이 곤란하다는 점을 설명하고, "다른 민원 처리를 위해 통화를 종료하오니 양해하여 주시기 바랍니다."라고 말합니다.  
STEP 4: 필요시, 통화 내용과 정도를 감안하여 위법행위 관리대장을 작성합니다.

3. 사용할 수 있는 안내 표현  
- "선생님, 말씀하신 민원은 정해진 규정과 절차에 따라 정상적으로 처리된 사항입니다."  
- "동일한 답변 외에는 도움을 드릴 수 없으므로 다른 민원 처리를 위해 통화를 종료하오니 양해하여 주시기 바랍니다."  
- "현재 처리 중에 있습니다."  

4. 근거  
- 충분한 설명이 이루어졌음에도 반복전화하는 경우, 통화를 종료해야 하며, 이는 공무방해행위에 해당할 수 있음을 경고해야 합니다. (근거 3)  
- 정당한 사유 없는 반복 전화는 동일 민원을 3회 이상 제기한 경우로, 부서장에게 보고하고 통화 지속이 곤란하다는 점을 설명한 후 상담을 종료해야 합니다. (근거 4)

5. 주의사항  
민원인과의 감정적 맞대응을 피하고, 침착하고 일관된 민원 응대를 유지해야 합니다. 또한, 반복 전화를 받는 부서원에게는 30분 이내의 휴게시간을 부여할 수 있습니다.


## 핵심 정리

```python
question
→ vectorstore.similarity_search(question)
→ format_docs(retrieved_docs)
→ prompt + llm
→ answer
```

- Naive RAG는 질문을 그대로 검색한다.
- 검색 품질이 낮으면 답변 품질도 낮아진다.
- 다음 단계에서는 검색 품질을 개선하는 Advanced RAG를 다룬다.